In [ ]:
### Visits one parliament meeting and extracts meta-data for all MP speeches.
import re
import uuid
import requests
from bs4 import BeautifulSoup

# Precompile a regex to strip leading "DD.MM.YYYY kl. HH:MM" style timestamps
_DATETIME_PREFIX_RE = re.compile(r"^\s*\d{2}\.\d{2}\.\d{4}\s+kl\.\s*\d{2}:\d{2}\s*")

# Regex for extracting date in DD.MM.YYYY format
_DATE_RE = re.compile(r"(\d{2})\.(\d{2})\.(\d{4})")
# Regex for extracting meeting_id from URL query
_MEETING_ID_RE = re.compile(r"id=(\d+)")

# Static metadata
_LANGUAGE = "Faroese"
_LOCATION = "Løgtingið (The Faroese Parliament)"
_AUDIO_FORMAT = "wav"
_SAMPLING_RATE = 16000

def parse_parliament(url):
    """
    Parse the Faroese parliament meeting page and extract metadata per speech.

    Args:
        url (str): URL of the meeting page, with an 'id' query parameter.

    Returns:
        List[dict]: A list of records containing meeting metadata and speech info.
    """
    # Extract meeting_id from URL
    id_match = _MEETING_ID_RE.search(url)
    meeting_id = id_match.group(1) if id_match else None

    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    current_date = None
    current_topic = None
    records = []

    # State variables for advanced topic assignment
    chairman_topic_active = False
    last_chairman_record_index = -1

    for li in soup.select("ul#playlist > li"):
        # Topic entries (identified by the 'py-2' class)
        if "py-2" in li.get("class", []):
            raw = li.get_text(strip=True)

            if "Fundur settur" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
            elif any(kw in raw for kw in ("Fundur niðurlagdur", "Fundur uppafturtikin")):
                continue
            else:
                # A new topic is being introduced
                if last_chairman_record_index != -1 and records[last_chairman_record_index].get('topic') is not None:
                    records[last_chairman_record_index]['topic'] = None

                topic_text = _DATETIME_PREFIX_RE.sub("", raw)
                current_topic = topic_text

                # Reset state for the new topic
                chairman_topic_active = False
                last_chairman_record_index = -1

        # Speech entries (all other list items)
        else:
            text = li.get_text(strip=True)
            m = re.match(r"(\d{2}:\d{2})\s*-\s*(.+)", text)
            if not m:
                continue
            time_str, name_part = m.groups()
            time_str = f"{time_str}:00"
            name = name_part.replace("(tf)", "").strip()

            if "(tf)" in name_part:
                contribution_type = "chairman"
            elif li.find("i", class_="fas fa-reply"):
                contribution_type = "remark"
            else:
                contribution_type = "speaker"

            seconds = None
            a_tag = li.find("a", onclick=True)
            if a_tag and "videoSeek" in a_tag.get("onclick", ""):
                sec_match = re.search(r"videoSeek\(\s*([\d\.]+)", a_tag["onclick"])
                if sec_match:
                    val = float(sec_match.group(1))
                    seconds = int(val) if val.is_integer() else val

            assigned_topic = None
            if contribution_type == "chairman":
                if chairman_topic_active:
                    assigned_topic = current_topic
            else:
                chairman_topic_active = True
                assigned_topic = current_topic

            record = {
                # MODIFICATION: Added a unique ID to each record
                "id":                str(uuid.uuid4()),
                "meeting_id":        meeting_id,
                "url":               url,
                "date":              current_date,
                "time":              time_str,
                "name":              name,
                "contribution_type": contribution_type,
                "second":            seconds,
                "topic":             assigned_topic,
                "language":          _LANGUAGE,
                "location":          _LOCATION,
                "audio_format":      _AUDIO_FORMAT,
                "sampling_rate":     _SAMPLING_RATE
            }
            records.append(record)

            if contribution_type == "chairman":
                last_chairman_record_index = len(records) - 1

    return records

if __name__ == "__main__":
    url = "https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556"
    for rec in parse_parliament(url):
        print(rec)

{'id': '8949eaae-443b-4715-8f63-5d5e588196c9', 'meeting_id': '7556', 'url': 'https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556', 'date': '2025-05-02', 'time': '09:35:00', 'name': 'Bjørt Samuelsen', 'contribution_type': 'chairman', 'second': 135, 'topic': None, 'language': 'Faroese', 'location': 'Løgtingið (The Faroese Parliament)', 'audio_format': 'wav', 'sampling_rate': 16000}
{'id': 'a95f5612-0913-40d9-9a6b-4c883932a23d', 'meeting_id': '7556', 'url': 'https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556', 'date': '2025-05-02', 'time': '09:37:00', 'name': 'Bjørt Samuelsen', 'contribution_type': 'chairman', 'second': 225, 'topic': None, 'language': 'Faroese', 'location': 'Løgtingið (The Faroese Parliament)', 'audio_format': 'wav', 'sampling_rate': 16000}
{'id': '71ef31cd-2902-4822-b298-f7fbd4554489', 'meeting_id': '7556', 'url': 'https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556', 'date': '2025-05-02', 'time': '09:40:00', 'name': 'Bjørt Samuelsen'

In [ ]:
## ### Visits all parliament meetings and extracts meta-data for speeches from a list of URL (txt file)

import re
import uuid
import requests
from bs4 import BeautifulSoup
import json

# Precompile a regex to strip leading "DD.MM.YYYY kl. HH:MM" style timestamps
_DATETIME_PREFIX_RE = re.compile(r"^\s*\d{2}\.\d{2}\.\d{4}\s+kl\.\s*\d{2}:\d{2}\s*")

# Regex for extracting date in DD.MM.YYYY format
_DATE_RE = re.compile(r"(\d{2})\.(\d{2})\.(\d{4})")
# Regex for extracting meeting_id from URL query
_MEETING_ID_RE = re.compile(r"id=(\d+)")

# Static metadata
_LANGUAGE = "Faroese"
_LOCATION = "Løgtingið (The Faroese Parliament)"
_AUDIO_FORMAT = "wav"
_SAMPLING_RATE = 16000

def parse_parliament(url):
    """
    Parse the Faroese parliament meeting page and extract metadata per speech.

    Args:
        url (str): URL of the meeting page, with an 'id' query parameter.

    Returns:
        List[dict]: A list of records containing meeting metadata and speech info.
    """
    # Extract meeting_id from URL
    id_match = _MEETING_ID_RE.search(url)
    meeting_id = id_match.group(1) if id_match else None

    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    current_date = None
    current_topic = None
    records = []

    # State variables for advanced topic assignment
    chairman_topic_active = False
    last_chairman_record_index = -1

    for li in soup.select("ul#playlist > li"):
        # Topic entries (identified by the 'py-2' class)
        if "py-2" in li.get("class", []):
            raw = li.get_text(strip=True)

            if "Fundur settur" or "Til viðgerðar" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
            elif any(kw in raw for kw in ("Fundur niðurlagdur", "Fundur uppafturtikin")):
                continue
            else:
                # A new topic is being introduced
                if last_chairman_record_index != -1 and records[last_chairman_record_index].get('topic') is not None:
                    records[last_chairman_record_index]['topic'] = None

                topic_text = _DATETIME_PREFIX_RE.sub("", raw)
                current_topic = topic_text

                # Reset state for the new topic
                chairman_topic_active = False
                last_chairman_record_index = -1

        # Speech entries (all other list items)
        else:
            text = li.get_text(strip=True)
            m = re.match(r"(\d{2}:\d{2})\s*-\s*(.+)", text)
            if not m:
                continue
            time_str, name_part = m.groups()
            time_str = f"{time_str}:00"
            name = name_part.replace("(tf)", "").strip()

            if "(tf)" in name_part:
                contribution_type = "chairman"
            elif li.find("i", class_="fas fa-reply"):
                contribution_type = "remark"
            else:
                contribution_type = "speaker"

            seconds = None
            a_tag = li.find("a", onclick=True)
            if a_tag and "videoSeek" in a_tag.get("onclick", ""):
                sec_match = re.search(r"videoSeek\(\s*([\d\.]+)", a_tag["onclick"])
                if sec_match:
                    val = float(sec_match.group(1))
                    seconds = int(val) if val.is_integer() else val

            assigned_topic = None
            if contribution_type == "chairman":
                if chairman_topic_active:
                    assigned_topic = current_topic
            else:
                chairman_topic_active = True
                assigned_topic = current_topic

            record = {
                "id":                str(uuid.uuid4()),
                "meeting_id":        meeting_id,
                "url":               url,
                "date":              current_date,
                "time":              time_str,
                "name":              name,
                "contribution_type": contribution_type,
                "second":            seconds,
                "topic":             assigned_topic,
                "language":          _LANGUAGE,
                "location":          _LOCATION,
                "audio_format":      _AUDIO_FORMAT,
                "sampling_rate":     _SAMPLING_RATE
            }
            records.append(record)

            if contribution_type == "chairman":
                last_chairman_record_index = len(records) - 1

    return records

def main():
    #input_file = "meeting_video_urls_2025-05-15.txt"
    input_file = "meeting_video_urls_TESTING_2025-06-20.txt"
    #output_file = "parsed_parliament_records_2025-06-23.txt"
    output_file = "parsed_parliament_records_TESTING_2025-06-23.txt"
    total_count = 0

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            first_url = line.split(',', 1)[0].strip()
            try:
                recs = parse_parliament(first_url)
                # Write each record as a JSON line
                for rec in recs:
                    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total_count += len(recs)
                print(f"✓ Saved {len(recs)} records from {first_url}")
            except Exception as e:
                print(f"✗ Error parsing {first_url}: {e}")

            # print("-" * 80)

    print(f"Done. Saved {total_count} records to '{output_file}'.")

if __name__ == "__main__":
    main()


✓ Saved 1039 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=2
Done. Saved 1039 records to 'parsed_parliament_records_TESTING_2025-06-23.txt'.


In [ ]:
### Visits all parliament meetings and extracts meta-data for speeches from a list of URL (txt file)
import re
import uuid
import requests
from bs4 import BeautifulSoup
import json

# Precompile a regex to strip leading "DD.MM.YYYY kl. HH:MM" style timestamps
_DATETIME_PREFIX_RE = re.compile(r"^\s*\d{2}\.\d{2}\.\d{4}\s+kl\.\s*\d{2}:\d{2}\s*")

# Regex for extracting date in DD.MM.YYYY format
_DATE_RE = re.compile(r"(\d{2})\.(\d{2})\.(\d{4})")
# Regex for extracting meeting_id from URL query
_MEETING_ID_RE = re.compile(r"id=(\d+)")

# Static metadata
_LANGUAGE = "Faroese"
_LOCATION = "Løgtingið (The Faroese Parliament)"
_AUDIO_FORMAT = "wav"
_SAMPLING_RATE = 16000

def parse_parliament(url):
    """
    Parse the Faroese parliament meeting page and extract metadata per speech.
    Handles multiple HTML structures for topic and date announcements.

    Args:
        url (str): URL of the meeting page, with an 'id' query parameter.

    Returns:
        List[dict]: A list of records containing meeting metadata and speech info.
    """
    # Extract meeting_id from URL
    id_match = _MEETING_ID_RE.search(url)
    meeting_id = id_match.group(1) if id_match else None

    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    current_date = None
    current_topic = None
    records = []

    # State variables for advanced topic assignment for the chairman
    chairman_topic_active = False
    last_chairman_record_index = -1

    for li in soup.select("ul#playlist > li"):
        # Topic entries are identified by the 'py-2' class
        if "py-2" in li.get("class", []):
            raw = li.get_text(strip=True)

            # Case 1: The meeting is formally starting.
            if "Fundur settur" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                current_topic = None # A new meeting start implies no topic yet
                chairman_topic_active = False
                last_chairman_record_index = -1

            # Case 2: A line that contains "Til viðgerðar". This confirms the date
            # for a topic that was likely announced in the *previous* <li> item.
            # We ONLY extract the date and do NOT set the topic.
            elif "Til viðgerðar" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"

            # Case 3: Pauses in the meeting. Ignore these.
            elif any(kw in raw for kw in ("Fundur niðurlagdur", "Fundur uppafturtikin")):
                continue

            # Case 4: This is a real topic header.
            else:
                # This is a new topic, so retrospectively nullify the last chairman's topic if set.
                if last_chairman_record_index != -1 and records and records[last_chairman_record_index].get('topic') is not None:
                     records[last_chairman_record_index]['topic'] = None

                # Clean the text to get the topic.
                topic_text = _DATETIME_PREFIX_RE.sub("", raw)

                # If a date is also present in this line, extract it.
                m = _DATE_RE.search(raw)
                if m:
                     current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"

                # Set the topic if the cleaned text is not empty.
                if topic_text:
                    current_topic = topic_text

                # Reset state variables for the new topic.
                chairman_topic_active = False
                last_chairman_record_index = -1

        # Speech entries (all other list items)
        else:
            text = li.get_text(strip=True)
            m = re.match(r"(\d{2}:\d{2})\s*-\s*(.+)", text)
            if not m:
                continue
            time_str, name_part = m.groups()
            time_str = f"{time_str}:00"
            name = name_part.replace("(tf)", "").strip()

            if "(tf)" in name_part:
                contribution_type = "chairman"
            elif li.find("i", class_="fas fa-reply"):
                contribution_type = "remark"
            else:
                contribution_type = "speaker"

            seconds = None
            a_tag = li.find("a", onclick=True)
            if a_tag and "videoSeek" in a_tag.get("onclick", ""):
                sec_match = re.search(r"videoSeek\(\s*([\d\.]+)", a_tag["onclick"])
                if sec_match:
                    val = float(sec_match.group(1))
                    seconds = int(val) if val.is_integer() else val

            assigned_topic = None
            if contribution_type == "chairman":
                if chairman_topic_active:
                    assigned_topic = current_topic
            else:
                chairman_topic_active = True
                assigned_topic = current_topic

            record = {
                "id":                str(uuid.uuid4()),
                "meeting_id":        meeting_id,
                "url":               url,
                "date":              current_date,
                "time":              time_str,
                "name":              name,
                "contribution_type": contribution_type,
                "second":            seconds,
                "topic":             assigned_topic,
                "language":          _LANGUAGE,
                "location":          _LOCATION,
                "audio_format":      _AUDIO_FORMAT,
                "sampling_rate":     _SAMPLING_RATE
            }
            records.append(record)

            if contribution_type == "chairman":
                last_chairman_record_index = len(records) - 1

    return records

def main():
    input_file = "meeting_video_urls_2025-05-15.txt"
    output_file = "parsed_parliament_records_2025-06-23_v2.txt"
    total_count = 0

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            first_url = line.split(',', 1)[0].strip()
            try:
                recs = parse_parliament(first_url)
                for rec in recs:
                    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total_count += len(recs)
                print(f"✓ Saved {len(recs)} records from {first_url}")
            except Exception as e:
                print(f"✗ Error parsing {first_url}: {e}")

    print(f"Done. Saved {total_count} records to '{output_file}'.")

if __name__ == "__main__":
    main()


✓ Saved 2 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7558
✓ Saved 3 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7557
✓ Saved 485 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556
✓ Saved 6 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7554
✓ Saved 742 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7553
✓ Saved 723 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7552
✓ Saved 442 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7550
✓ Saved 389 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7548
✓ Saved 550 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7545
✓ Saved 1 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7542
✓ Saved 388 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7540
✓ Saved 339 records from http

In [ ]:
### Visits all parliament meetings and extracts meta-data for speeches from a list of URL (txt file)
### THIS IS THE FINAL WORKING VERSION

import re
import uuid
import requests
from bs4 import BeautifulSoup
import json

# Precompile a regex to strip leading "DD.MM.YYYY kl. HH:MM" style timestamps
_DATETIME_PREFIX_RE = re.compile(r"^\s*\d{2}\.\d{2}\.\d{4}\s+kl\.\s*\d{2}:\d{2}\s*")

# Regex for extracting date in DD.MM.YYYY format
_DATE_RE = re.compile(r"(\d{2})\.(\d{2})\.(\d{4})")
# Regex for extracting meeting_id from URL query
_MEETING_ID_RE = re.compile(r"id=(\d+)")

# Static metadata
_LANGUAGE = "Faroese"
_LOCATION = "Løgtingið (The Faroese Parliament)"
_AUDIO_FORMAT = "wav"
_SAMPLING_RATE = 16000

def parse_parliament(url):
    """
    Parse the Faroese parliament meeting page and extract metadata per speech.
    Handles multiple HTML structures and pre-assigns topics to chairmen introductions.

    Args:
        url (str): URL of the meeting page, with an 'id' query parameter.

    Returns:
        List[dict]: A list of records containing meeting metadata and speech info.
    """
    # Extract meeting_id from URL
    id_match = _MEETING_ID_RE.search(url)
    meeting_id = id_match.group(1) if id_match else None

    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    current_date = None
    current_topic = None
    records = []

    # Get all list items to enable look-ahead
    all_lis = soup.select("ul#playlist > li")

    for i, li in enumerate(all_lis):
        # Topic entries are identified by the 'py-2' class
        if "py-2" in li.get("class", []):
            raw = li.get_text(strip=True)

            if "Fundur settur" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                current_topic = None

            elif "Til viðgerðar" in raw:
                m = _DATE_RE.search(raw)
                if m:
                    current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"

            elif any(kw in raw for kw in ("Fundur niðurlagdur", "Fundur uppafturtikin")):
                continue

            # This is a real topic header.
            else:
                topic_text = _DATETIME_PREFIX_RE.sub("", raw)
                m = _DATE_RE.search(raw)
                if m:
                     current_date = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                if topic_text:
                    current_topic = topic_text

        # Speech entries (all other list items)
        else:
            text = li.get_text(strip=True)
            m = re.match(r"(\d{2}:\d{2})\s*-\s*(.+)", text)
            if not m:
                continue
            time_str, name_part = m.groups()
            time_str = f"{time_str}:00"
            name = name_part.replace("(tf)", "").strip()

            if "(tf)" in name_part:
                contribution_type = "chairman"
            elif li.find("i", class_="fas fa-reply"):
                contribution_type = "remark"
            else:
                contribution_type = "speaker"

            seconds = None
            a_tag = li.find("a", onclick=True)
            if a_tag and "videoSeek" in a_tag.get("onclick", ""):
                sec_match = re.search(r"videoSeek\(\s*([\d\.]+)", a_tag["onclick"])
                if sec_match:
                    val = float(sec_match.group(1))
                    seconds = int(val) if val.is_integer() else val

            # --- MODIFICATION: Topic assignment with look-ahead for chairman ---
            assigned_topic = current_topic # Start with the current topic by default

            if contribution_type == "chairman":
                # Look ahead to the next list item to see if it's a new topic
                if i + 1 < len(all_lis):
                    next_li = all_lis[i+1]
                    # Check if the next item is a topic header
                    if "py-2" in next_li.get("class", []):
                        next_raw_text = next_li.get_text(strip=True)
                        # Check if it's a *real* topic header and not just a status update
                        if not any(kw in next_raw_text for kw in ("Fundur settur", "Til viðgerðar", "Fundur niðurlagdur", "Fundur uppafturtikin")):
                            # If so, this chairman is introducing the next topic. Assign it now.
                            future_topic_text = _DATETIME_PREFIX_RE.sub("", next_raw_text)
                            assigned_topic = future_topic_text

            record = {
                "id":                str(uuid.uuid4()),
                "meeting_id":        meeting_id,
                "url":               url,
                "date":              current_date,
                "time":              time_str,
                "name":              name,
                "contribution_type": contribution_type,
                "second":            seconds,
                "topic":             assigned_topic,
                "language":          _LANGUAGE,
                "location":          _LOCATION,
                "audio_format":      _AUDIO_FORMAT,
                "sampling_rate":     _SAMPLING_RATE
            }
            records.append(record)

    return records

def main():
    input_file = "meeting_video_urls_2025-05-15.txt"
    output_file = "parsed_parliament_records_2025-06-23_v3.txt"
    total_count = 0

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            first_url = line.split(',', 1)[0].strip()
            try:
                recs = parse_parliament(first_url)
                for rec in recs:
                    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total_count += len(recs)
                print(f"✓ Saved {len(recs)} records from {first_url}")
            except Exception as e:
                print(f"✗ Error parsing {first_url}: {e}")

    print(f"Done. Saved {total_count} records to '{output_file}'.")

if __name__ == "__main__":
    main()


✓ Saved 2 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7558
✓ Saved 3 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7557
✓ Saved 485 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556
✓ Saved 6 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7554
✓ Saved 742 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7553
✓ Saved 723 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7552
✓ Saved 442 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7550
✓ Saved 389 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7548
✓ Saved 550 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7545
✓ Saved 1 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7542
✓ Saved 388 records from https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7540
✓ Saved 339 records from http

In [ ]:
### Enriches parliament records (faroese_mps_2020_2025_meta-data) with data from an MP profiles file and an MP metadata CSV files
### THIS IS THE FINAL WORKING VERSION

import json
import csv
import re

def normalize_name(name):
    """
    Normalizes a name by replacing multiple spaces with a single space
    and trimming leading/trailing whitespace.

    Args:
        name (str): The name string to normalize.

    Returns:
        str: The normalized name.
    """
    if not isinstance(name, str):
        return ""
    # Replace multiple whitespace characters with a single space
    name = re.sub(r'\s+', ' ', name)
    # Remove leading/trailing whitespace
    return name.strip()

def enrich_parliament_records(parliament_records_file, mp_profiles_file, mp_metadata_file, output_file):
    """
    Enriches parliament records with data from an MP profiles file and an MP metadata CSV file.
    It normalizes names for matching and reports any names that are not found.

    Args:
        parliament_records_file (str): Path to the file with parsed parliament records.
        mp_profiles_file (str): Path to the file with MP profiles (JSONL format).
        mp_metadata_file (str): Path to the CSV file with MP metadata.
        output_file (str): Path to the output file for the enriched records.
    """
    # 1. Load MP profiles (ID and URL) using normalized names as keys
    mp_profiles_data = {}
    try:
        with open(mp_profiles_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    profile = json.loads(line)
                    if 'name' in profile and 'id' in profile and 'url' in profile:
                        clean_name = normalize_name(profile['name'])
                        if clean_name:
                            mp_profiles_data[clean_name] = {
                                'mp_id': profile['id'],
                                'mp_url': profile['url']
                            }
                except json.JSONDecodeError:
                    print(f"Warning: Could not decode JSON from line in {mp_profiles_file}: {line.strip()}")
    except FileNotFoundError:
        print(f"Error: The file {mp_profiles_file} was not found.")
        return

    # 2. Load MP metadata from the CSV file using normalized names as keys
    mp_metadata = {}
    try:
        with open(mp_metadata_file, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                original_name = row.get('name')
                if original_name:
                    clean_name = normalize_name(original_name)
                    if clean_name:
                        mp_metadata[clean_name] = row
    except FileNotFoundError:
        print(f"Error: The file {mp_metadata_file} was not found.")
        return
    except Exception as e:
        print(f"An error occurred while reading {mp_metadata_file}: {e}")
        return

    # Sets to track names that are not found in the enrichment files
    unmatched_in_profiles = set()
    unmatched_in_metadata = set()

    # 3. Process the parliament records, enrich data, and write to the output file
    try:
        with open(parliament_records_file, 'r', encoding='utf-8') as infile, \
             open(output_file, 'w', encoding='utf-8') as outfile:
            for line in infile:
                try:
                    record = json.loads(line)
                    original_name = record.get('name')

                    if original_name:
                        # Normalize the name from the main record for lookup
                        clean_name = normalize_name(original_name)

                        # Add data from mp_profiles_data
                        if clean_name in mp_profiles_data:
                            record.update(mp_profiles_data[clean_name])
                        else:
                            unmatched_in_profiles.add(clean_name)

                        # Add data from mp_metadata
                        if clean_name in mp_metadata:
                            metadata = mp_metadata[clean_name].copy()
                            # We can remove the redundant 'name' key from the metadata
                            if 'name' in metadata:
                                del metadata['name']
                            record.update(metadata)
                        else:
                            unmatched_in_metadata.add(clean_name)

                    outfile.write(json.dumps(record, ensure_ascii=False) + '\n')

                except json.JSONDecodeError:
                    print(f"Warning: Could not decode JSON from line in {parliament_records_file}: {line.strip()}")

        print("-" * 50)
        print(f"Successfully created enriched file: {output_file}")
        print("-" * 50)

    except FileNotFoundError:
        print(f"Error: The file {parliament_records_file} was not found.")
        return

    # 4. Report any unmatched names
    if unmatched_in_profiles or unmatched_in_metadata:
        print("\nData Matching Report:")
        print("The following MPs from the parliament records could not be found in the enrichment files.")

        if unmatched_in_profiles:
            print(f"\nMPs not found in '{mp_profiles_file}':")
            for name in sorted(list(unmatched_in_profiles)):
                print(f"  - {name}")

        if unmatched_in_metadata:
            print(f"\nMPs not found in '{mp_metadata_file}':")
            for name in sorted(list(unmatched_in_metadata)):
                print(f"  - {name}")
    else:
        print("\nData Matching Report: All MPs were successfully matched in all files.")


if __name__ == '__main__':
    # Updated file names as per your request
    parliament_records_file = 'parsed_parliament_records_2025-06-23_v3.txt'
    mp_profiles_file = 'mp_profiles_all.txt'
    mp_metadata_file = 'faroese_mps_2020_2025_meta-data_2025-06-24.csv'
    output_file = 'parliament_meta_data_2025-06_24_v2.jsonl'

    enrich_parliament_records(parliament_records_file, mp_profiles_file, mp_metadata_file, output_file)

--------------------------------------------------
Successfully created enriched file: parliament_meta_data_2025-06_24_v2.jsonl
--------------------------------------------------

Data Matching Report: All MPs were successfully matched in all files.


In [ ]:
!pip install pydub

In [ ]:
print("hi")

hi


In [ ]:
## Genereast new file with MP names from list of URL's (parlament meetings)
## input is txt file with urls (meeting_video_urls.txt)

import re
import requests
from bs4 import BeautifulSoup

def fetch_playlist(url):
    """
    Fetch the webpage and extract playlist data from the <ul id="playlist">.
    Returns a list of dictionaries with:
      - event: text content of the list element.
      - second: time (in seconds) extracted from videoSeek() if available.
      - video_id: video id extracted from videoSeek() if available.
    """
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    playlist_ul = soup.find('ul', id='playlist')
    if playlist_ul is None:
        print("No playlist found on the page.")
        return []

    playlist_data = []
    for li in playlist_ul.find_all('li'):
        event_text = li.get_text(strip=True)
        seconds = None
        video_id = None
        a_tag = li.find('a', onclick=True)
        if a_tag and 'videoSeek' in a_tag.get('onclick', ''):
            match = re.search(r'videoSeek\(\s*([\d\.]+)\s*,\s*[\'"]?([^\'"\)]+)[\'"]?\s*\)', a_tag['onclick'])
            if match:
                try:
                    sec_val = float(match.group(1))
                    seconds = int(sec_val) if sec_val.is_integer() else sec_val
                except ValueError:
                    seconds = match.group(1)
                raw_video_id = match.group(2)
                digits = ''.join(filter(str.isdigit, raw_video_id))
                try:
                    video_id = int(digits) if digits else None
                except ValueError:
                    video_id = None
        playlist_data.append({
            "event": event_text,
            "second": seconds,
            "video_id": video_id
        })
    return playlist_data

def extract_mp_names(playlist_data):
    """
    Process the playlist data to extract MP names.
    Uses the same logic as align_asr_to_events: if the event text starts with a time (hh:mm -),
    the remainder is considered the MP name.
    Excludes names that contain any digits or include any of the keywords:
    "Formaðurin", "Framløga", "Fundur", "Uppskot", "viðgerðar", "Farloyvi".
    """
    excluded_keywords = ["Formaðurin", "Framløga", "Fundur", "Uppskot", "viðgerðar",
                         "Farloyvi", "Feilurin", "Felagsforeldramyndugleiki", "Føroyskir",
                         "Góðkenning", "undirsjóvarbreyt", "Negativa", "Bolognatilgongdin",
                         "Broyting", "Eftirsleipið", "Eldraverja", "Endurskoðan", "Fiskirættindi",
                         "Fosturtøkur", "Framtíðarstøða", "Fráboðan", "Fullveldisskjøl",
                         "Heimsheilsustovnurin", "Inflasjón", "Laksaroyndir", "Marknaðaratgongd",
                         "Minningarorð", "heimatænasta", "Nýskipanir", "bjargingarskip",
                         "Oljuleiting", "Palliatión", ]
    names = set()
    for event in playlist_data:
        raw_event = event["event"]
        # If the event starts with a time, extract the name after the time.
        match = re.match(r"^(\d\d:\d\d)\s*-\s*(.+)$", raw_event)
        if match:
            name_val = match.group(2)
        else:
            name_val = raw_event

        # Exclude names containing any digit.
        if re.search(r'\d', name_val):
            continue

        # Exclude names containing any of the keywords.
        if any(keyword in name_val for keyword in excluded_keywords):
            continue

        names.add(name_val)
    return names

def get_mp_names_from_url(url):
    """
    Given a URL of a parliament meeting page, fetch the playlist data and extract MP names.
    """
    playlist_data = fetch_playlist(url)
    return extract_mp_names(playlist_data)

def main():
    input_file = 'meeting_video_urls_2025-05-15.txt'  # Input file with two comma-separated URLs per line.
    output_file = 'mp_names.txt'
    unique_mp_names = set()

    with open(input_file, 'r') as f:
        for line in f:
            if line.strip():
                first_url = line.split(',')[0].strip()
                print(f"Processing URL: {first_url}")
                try:
                    names = get_mp_names_from_url(first_url)
                    unique_mp_names.update(names)
                except Exception as e:
                    print(f"Error processing {first_url}: {e}")

    with open(output_file, 'w') as f:
        for name in sorted(unique_mp_names):
            f.write(name + "\n")
    print(f"Collected {len(unique_mp_names)} unique MP names in '{output_file}'.")

if __name__ == "__main__":
    main()


Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7558
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7557
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7556
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7554
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7553
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7552
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7550
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7548
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7545
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7542
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7540
Processing URL: https://www.logting.fo/mal/yvirlit/gerdabokur/gerdabok/?id=7537
Processing URL: https://www.logting.fo/m

In [ ]:
#### Gets names and id's from MP's profile page from a list of URLs (text file "mp_profiles.txt")

import requests
from bs4 import BeautifulSoup
import json
import time

BASE_DOMAIN = "https://www.logting.fo"
BASE_URL = f"{BASE_DOMAIN}/tingfolk-og-nevndir/aktorur/"
OUTPUT_FILE = "mp_profiles.txt"
MAX_PROFILE_ID = 192 # this is currently the newest profile
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/90.0.4430.93 Safari/537.36"
    )
}

profiles = []

for mp_id in range(1, MAX_PROFILE_ID):
    page_url = f"{BASE_URL}?id={mp_id}"
    try:
        resp = requests.get(page_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Failed to fetch main page for ID {mp_id}: {e}")
        continue

    soup = BeautifulSoup(resp.text, 'html.parser')
    container = soup.find('div', class_='macroContainer')
    if not container or not container.has_attr('data-url'):
        print(f"No macroContainer/data-url for ID {mp_id}. URL: {page_url}")
        continue

    detail_path = container['data-url']
    # Build full URL for actor detail
    detail_url = detail_path if detail_path.startswith('http') else f"{BASE_DOMAIN}{detail_path}"

    try:
        detail_resp = requests.get(detail_url, headers=HEADERS, timeout=10)
        detail_resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Failed to fetch detail for ID {mp_id}: {e}")
        continue

    detail_soup = BeautifulSoup(detail_resp.text, 'html.parser')
    name_tag = detail_soup.select_one('div.person-detail h2')
    if not name_tag:
        print(f"Name tag not found in detail for ID {mp_id}. URL: {detail_url}")
        continue

    name = name_tag.get_text(strip=True)
    formatted_id = f"MPID{mp_id:06d}"

    profiles.append({
        "name": name,
        "id": formatted_id,
        "url": page_url
    })

    # polite pause
    time.sleep(0.5)

# Write results
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for p in profiles:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Scraped {len(profiles)} profiles. Output written to {OUTPUT_FILE}.")


Scraped 191 profiles. Output written to mp_profiles.txt.


In [ ]:
import json
import re
from datetime import datetime # Import the datetime module

def update_and_sort_jsonl(input_file_path):
    """
    Reads a JSONL file, updates the 'id' field for correct sorting,
    sorts the data by the new 'id', and writes it to a new file
    with a timestamp in the filename.

    Args:
        input_file_path (str): The path to the input JSONL file.
    """
    # --- MODIFICATION START ---
    # Get the current date and time in a file-friendly format
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    # Safely get the filename without the '.jsonl' extension
    base_name = input_file_path.rsplit('.jsonl', 1)[0]

    # Define the output file path by appending '_sorted_' and the timestamp
    output_file_path = f"{base_name}_sorted_{timestamp}.jsonl"
    # --- MODIFICATION END ---

    processed_lines = []

    try:
        # Open and read the input file line by line
        with open(input_file_path, 'r', encoding='utf-8') as f_in:
            for line in f_in:
                # Load each line as a JSON object
                data = json.loads(line)
                original_id = data.get("id")

                if original_id:
                    # Use a regular expression to extract parts of the ID
                    match = re.match(r'M(\d+)_(S\d+)', original_id)
                    if match:
                        meeting_number = int(match.group(1))
                        session_part = match.group(2)

                        # Format the meeting number to be 4 digits with leading zeros
                        new_id = f"M{meeting_number:04d}_{session_part}"

                        # Update the 'id' in the dictionary
                        data['id'] = new_id

                processed_lines.append(data)

        # Sort the list of dictionaries based on the new 'id'
        sorted_data = sorted(processed_lines, key=lambda x: x.get('id', ''))

        # Write the sorted data to the new output file
        with open(output_file_path, 'w', encoding='utf-8') as f_out:
            for item in sorted_data:
                # Write each dictionary as a JSON string on a new line
                f_out.write(json.dumps(item, ensure_ascii=False) + '\n')

        print(f"✅ Successfully processed the file.")
        print(f"Sorted data has been saved to: {output_file_path}")

    except FileNotFoundError:
        print(f"❌ Error: The file '{input_file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# --- How to run the script ---
# 1. Save the code above as a Python file (e.g., `process_data_v2.py`).
# 2. Make sure the script is in the same directory as your .jsonl file.
# 3. Run the script from your terminal.

# Name of your input file
file_name = "PARLIAMENT_META_DATA_audio_id_2025-06-25_21-12-08.jsonl"

# Call the function to process the file
update_and_sort_jsonl(file_name)

✅ Successfully processed the file.
Sorted data has been saved to: PARLIAMENT_META_DATA_audio_id_2025-06-25_21-12-08_sorted_2025-06-30_13-20-24.jsonl


In [ ]:
print("#")

#
